# AIEngineer

## Enviroiment
``` bash
pip install "numpy<1.25" "pandas<2.0" "scipy<1.11" "scikit-learn<1.6" "matplotlib<3.8"
```

In [ ]:
import numpy as np
import scipy
import sklearn

print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)

# Простая проверка работы
from sklearn.linear_model import LinearRegression
model = LinearRegression()
print("LinearRegression OK")


## DAY 2

### CSV

In [ ]:
import pandas as pd

# Путь относительно корня проекта
df = pd.read_csv('data_pipeline/WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(df.shape)  # Сколько строк и колонок
print(df.head()) # Первые 5 строк
print(df.info()) # Типы данныx и пропуски

### Pandas

In [ ]:
import pandas as pd
df = pd.read_csv('data_pipeline/WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [ ]:
# 1. Фильтрация (WHERE)
df_filtered = df[df['tenure'] > 12].copy()

# 2. Флаг (CASE WHEN)
df_filtered.loc[:, 'is_high_value'] = (df_filtered['MonthlyCharges'] > 70).astype(int)

# 3. Оконная агрегация (AVG OVER PARTITION BY)
df_filtered['avg_monthly_by_contract'] = df_filtered.groupby('Contract')['MonthlyCharges'].transform('mean')

print(df_filtered[['customerID', 'tenure', 'MonthlyCharges', 'is_high_value', 'avg_monthly_by_contract']].head())

### Задача №1: привести типы и обработать «грязные» данные (твоя зона комфорта)

In [ ]:
# 1. Превращаем TotalCharges в float, пустые строки — в NaN
# Это аналог TRY_CAST / COALESCE в SQL
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# 2. Заполняем пропуски (если они появятся после конвертации)
# Аналог COALESCE(TotalCharges, 0)
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# 3. Переводим целевой признак Churn в 0/1
# CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END
df['ChurnFlag'] = (df['Churn'] == 'Yes').astype(int)

print(df[['TotalCharges', 'ChurnFlag']].info())
print(df[['TotalCharges', 'ChurnFlag']].describe())

### Задача №2: фичи, которые ты делал в SQL (оконные и агрегаты)

In [ ]:
import numpy as np

In [ ]:
# 1. Доля TotalCharges от суммы по типу контракта
# SQL: SUM(TotalCharges) OVER (PARTITION BY Contract)
sum_by_contract = df.groupby('Contract')['TotalCharges'].transform('sum')
df['ShareOfContract'] = df['TotalCharges'] / sum_by_contract

# Чтобы не было деления на ноль, если сумма 0
df['ShareOfContract'] = df['ShareOfContract'].replace([np.inf, -np.inf], 0).fillna(0)

# 2. Флаг: MonthlyCharges выше среднего по контракту
avg_monthly_by_contract = df.groupby('Contract')['MonthlyCharges'].transform('mean')
df['AboveAvgMonthlyByContract'] = (df['MonthlyCharges'] > avg_monthly_by_contract).astype(int)

print(df[['Contract', 'TotalCharges', 'ShareOfContract', 'MonthlyCharges', 'AboveAvgMonthlyByContract']].head(10))